In [0]:
from pyspark.sql import functions as F, Window

wm = spark.sql("""SELECT last_ingest_ts FROM workspace.default.etl_watermark
                  WHERE pipeline = 'silver_complaints'""").first()[0]
print("Last processed ingest_ts:", wm)

new = spark.table("workspace.default.bronze_complaints").filter(F.col("ingest_ts") > F.lit(wm))
batch_max = new.agg(F.max("ingest_ts")).first()[0]
print("Rows in this batch:", new.count())

if batch_max is None:
    dbutils.notebook.exit("No new rows - nothing to merge")

In [0]:
text_cols = ["product","sub_product","issue","sub_issue","company","state",
             "submitted_via","company_response_to_consumer","company_public_response","tags"]
for c in text_cols:
    new = new.withColumn(c, F.when(F.trim(F.col(c)).isin("", "None", "N/A"), None)
                             .otherwise(F.trim(F.col(c))))

new = (new
    .withColumn("complaint_id", F.col("complaint_id").cast("bigint"))
    .withColumn("date_received", F.to_date("date_received"))
    .withColumn("date_sent_to_company", F.to_date("date_sent_to_company"))
    .withColumn("timely_response_flag", F.when(F.col("timely_response") == "Yes", True)
                                         .when(F.col("timely_response") == "No", False))
    .withColumn("company", F.upper("company"))
    .withColumn("state", F.upper("state"))
    .withColumn("days_to_company", F.datediff("date_sent_to_company", "date_received"))
    .withColumn("load_ts", F.current_timestamp())
    .drop("source_file", "ingest_ts", "_rescued_data"))

w = Window.partitionBy("complaint_id").orderBy(F.col("date_received").desc())
new = new.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
new.createOrReplaceTempView("daily_batch")

In [0]:
spark.sql("""
MERGE INTO workspace.default.silver_complaints AS t
USING daily_batch AS s
  ON t.complaint_id = s.complaint_id
WHEN MATCHED AND ((t.company_response_to_consumer <=> s.company_response_to_consumer) = false
               OR (t.timely_response <=> s.timely_response) = false)
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""").show()

spark.sql(f"""
UPDATE workspace.default.etl_watermark
SET last_ingest_ts = '{batch_max}'
WHERE pipeline = 'silver_complaints'
""")
print("Watermark moved to:", batch_max)